# Conditional Probabilty


**Question asked:** <span style="color:red;">"How does the probability of one variable change after observing another variable"</span>

**Mathematical Definition:** $$P(X=x|Y=y)=\frac{P(X=x, Y=y)}{P(Y=y)}$$

where, $P(Y=y)$ obtained through marginalization

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Hashable, Sequence

import matplotlib.pyplot as plt
import numpy as np

In [4]:
@dataclass
class DiscreteDistribution:
    """
    Discrete probability distribution over one random variable.

    Parameters
    ----------
    values:
        Possible outcomes of the random variable.

    probabilities:
        Probability assigned to each outcome.

    variable_name:
        Optional name used for display and plotting.
    """

    values: Sequence[Hashable]
    probabilities: Sequence[float]
    variable_name: str = "X"

    def __post_init__(self) -> None:
        self.values = np.asarray(self.values, dtype=object)
        self.probabilities = np.asarray(
            self.probabilities,
            dtype=float,
        )

        self._validate_parameters()

        self._value_to_index = {
            value: index
            for index, value in enumerate(self.values)
        }

    def _validate_parameters(self) -> None:
        if self.values.ndim != 1:
            raise ValueError("values must be one-dimensional.")

        if self.probabilities.ndim != 1:
            raise ValueError(
                "probabilities must be one-dimensional."
            )

        if len(self.values) == 0:
            raise ValueError("values cannot be empty.")

        if len(self.values) != len(self.probabilities):
            raise ValueError(
                "values and probabilities must have the same length."
            )

        if len(set(self.values)) != len(self.values):
            raise ValueError("values must contain unique elements.")

        if not np.all(np.isfinite(self.probabilities)):
            raise ValueError(
                "probabilities must contain finite values."
            )

        if np.any(self.probabilities < 0.0):
            raise ValueError(
                "probabilities cannot be negative."
            )

        total_probability = self.probabilities.sum()

        if not np.isclose(total_probability, 1.0):
            raise ValueError(
                "probabilities must sum to 1. "
                f"Received sum={total_probability:.6f}."
            )

    def pmf(self, value: Hashable) -> float:
        """Return P(X=value)."""

        if value not in self._value_to_index:
            raise ValueError(
                f"Unknown value for {self.variable_name}: {value!r}."
            )

        index = self._value_to_index[value]

        return float(self.probabilities[index])

    def sample(
        self,
        size: int = 1,
        rng: np.random.Generator | None = None,
    ) -> np.ndarray:
        """Draw samples from the distribution."""

        if not isinstance(size, (int, np.integer)):
            raise TypeError("size must be an integer.")

        if size <= 0:
            raise ValueError("size must be greater than zero.")

        if rng is None:
            rng = np.random.default_rng()

        indices = rng.choice(
            len(self.values),
            size=size,
            p=self.probabilities,
        )

        return self.values[indices]

    def plot(self) -> None:
        """Plot the probability mass function."""

        fig, ax = plt.subplots(figsize=(7, 4))

        positions = np.arange(len(self.values))

        ax.bar(
            positions,
            self.probabilities,
        )

        ax.set_xticks(positions)
        ax.set_xticklabels(self.values)

        ax.set_xlabel(self.variable_name)
        ax.set_ylabel("Probability")
        ax.set_title(
            f"Marginal Distribution of {self.variable_name}"
        )

        ax.set_ylim(
            0.0,
            max(1.0, self.probabilities.max() * 1.15),
        )

        for position, probability in zip(
            positions,
            self.probabilities,
        ):
            ax.text(
                position,
                probability,
                f"{probability:.2f}",
                ha="center",
                va="bottom",
            )

        plt.tight_layout()
        plt.show()

In [10]:
@dataclass
class JointDistribution:
    """
    Discrete joint probability distribution over two variables.

    probabilities[i, j] represents

        P(X=x_values[i], Y=y_values[j])
    """

    x_values: Sequence[Hashable]
    y_values: Sequence[Hashable]
    probabilities: Sequence[Sequence[float]]
    x_name: str = "X"
    y_name: str = "Y"

    def __post_init__(self) -> None:
        self.x_values = np.asarray(
            self.x_values,
            dtype=object,
        )

        self.y_values = np.asarray(
            self.y_values,
            dtype=object,
        )

        self.probabilities = np.asarray(
            self.probabilities,
            dtype=float,
        )

        self._validate_parameters()

        self._x_to_index = {
            value: index
            for index, value in enumerate(self.x_values)
        }

        self._y_to_index = {
            value: index
            for index, value in enumerate(self.y_values)
        }

    def _validate_parameters(self) -> None:
        if self.x_values.ndim != 1:
            raise ValueError(
                "x_values must be one-dimensional."
            )

        if self.y_values.ndim != 1:
            raise ValueError(
                "y_values must be one-dimensional."
            )

        if len(self.x_values) == 0:
            raise ValueError("x_values cannot be empty.")

        if len(self.y_values) == 0:
            raise ValueError("y_values cannot be empty.")

        if len(set(self.x_values)) != len(self.x_values):
            raise ValueError(
                "x_values must contain unique values."
            )

        if len(set(self.y_values)) != len(self.y_values):
            raise ValueError(
                "y_values must contain unique values."
            )

        expected_shape = (
            len(self.x_values),
            len(self.y_values),
        )

        if self.probabilities.shape != expected_shape:
            raise ValueError(
                "Probability table must have shape "
                f"{expected_shape}, but received "
                f"{self.probabilities.shape}."
            )

        if not np.all(np.isfinite(self.probabilities)):
            raise ValueError(
                "Probabilities must contain finite values."
            )

        if np.any(self.probabilities < 0.0):
            raise ValueError(
                "Probabilities cannot be negative."
            )

        total_probability = self.probabilities.sum()

        if not np.isclose(total_probability, 1.0):
            raise ValueError(
                "Joint probabilities must sum to 1. "
                f"Received sum={total_probability:.6f}."
            )

    def pmf(
        self,
        x: Hashable,
        y: Hashable,
    ) -> float:
        """Return P(X=x, Y=y)."""

        if x not in self._x_to_index:
            raise ValueError(
                f"Unknown value for {self.x_name}: {x!r}."
            )

        if y not in self._y_to_index:
            raise ValueError(
                f"Unknown value for {self.y_name}: {y!r}."
            )

        x_index = self._x_to_index[x]
        y_index = self._y_to_index[y]

        return float(
            self.probabilities[x_index, y_index]
        )

    def marginal(
        self,
        variable: str,
    ) -> DiscreteDistribution:
        """
        Compute the marginal distribution of X or Y.

        Parameters
        ----------
        variable:
            Either "x" or "y".

        Returns
        -------
        DiscreteDistribution
            The marginal distribution of the selected variable.
        """

        variable = variable.lower()

        if variable == "x":
            # Remove Y by summing over axis 1.
            marginal_probabilities = (
                self.probabilities.sum(axis=1)
            )

            return DiscreteDistribution(
                values=self.x_values,
                probabilities=marginal_probabilities,
                variable_name=self.x_name,
            )

        if variable == "y":
            # Remove X by summing over axis 0.
            marginal_probabilities = (
                self.probabilities.sum(axis=0)
            )

            return DiscreteDistribution(
                values=self.y_values,
                probabilities=marginal_probabilities,
                variable_name=self.y_name,
            )

        raise ValueError(
            "variable must be either 'x' or 'y'."
        )
        
    def conditional(
        self,
        target: str,
        given_value: Hashable,
    ) -> DiscreteDistribution:
        """
        Compute a conditional probability distribution.
    
        Parameters
        ----------
        target:
            Variable whose conditional distribution is required.
    
            target="x" computes P(X | Y=given_value).
            target="y" computes P(Y | X=given_value).
    
        given_value:
            Observed value of the conditioning variable.
    
        Returns
        -------
        DiscreteDistribution
            The normalized conditional distribution.
        """
    
        target = target.lower()
    
        if target == "x":
            # We want P(X | Y=y).
            if given_value not in self._y_to_index:
                raise ValueError(
                    f"Unknown value for {self.y_name}: "
                    f"{given_value!r}."
                )
    
            y_index = self._y_to_index[given_value]
    
            # Select all X values corresponding to the observed Y.
            joint_slice = self.probabilities[:, y_index]
    
            # P(Y=y)
            evidence_probability = joint_slice.sum()
    
            if np.isclose(evidence_probability, 0.0):
                raise ValueError(
                    "Conditional probability is undefined because "
                    f"P({self.y_name}={given_value!r}) is zero."
                )
    
            conditional_probabilities = (
                joint_slice / evidence_probability
            )
    
            return DiscreteDistribution(
                values=self.x_values,
                probabilities=conditional_probabilities,
                variable_name=(
                    f"{self.x_name} | "
                    f"{self.y_name}={given_value}"
                ),
            )
    
        if target == "y":
            # We want P(Y | X=x).
            if given_value not in self._x_to_index:
                raise ValueError(
                    f"Unknown value for {self.x_name}: "
                    f"{given_value!r}."
                )
    
            x_index = self._x_to_index[given_value]
    
            # Select all Y values corresponding to the observed X.
            joint_slice = self.probabilities[x_index, :]
    
            # P(X=x)
            evidence_probability = joint_slice.sum()
    
            if np.isclose(evidence_probability, 0.0):
                raise ValueError(
                    "Conditional probability is undefined because "
                    f"P({self.x_name}={given_value!r}) is zero."
                )
    
            conditional_probabilities = (
                joint_slice / evidence_probability
            )
    
            return DiscreteDistribution(
                values=self.y_values,
                probabilities=conditional_probabilities,
                variable_name=(
                    f"{self.y_name} | "
                    f"{self.x_name}={given_value}"
                ),
            )
    
        raise ValueError(
            "target must be either 'x' or 'y'."
        )    

    def plot_heatmap(self) -> None:
        """Plot the joint probability distribution."""

        fig, ax = plt.subplots(figsize=(7, 5))

        image = ax.imshow(self.probabilities)

        ax.set_xticks(
            np.arange(len(self.y_values))
        )

        ax.set_yticks(
            np.arange(len(self.x_values))
        )

        ax.set_xticklabels(self.y_values)
        ax.set_yticklabels(self.x_values)

        ax.set_xlabel(self.y_name)
        ax.set_ylabel(self.x_name)
        ax.set_title("Joint Probability Distribution")

        for i in range(len(self.x_values)):
            for j in range(len(self.y_values)):
                ax.text(
                    j,
                    i,
                    f"{self.probabilities[i, j]:.2f}",
                    ha="center",
                    va="center",
                )

        fig.colorbar(
            image,
            ax=ax,
            label="Probability",
        )

        plt.tight_layout()
        plt.show()

In [8]:
weather_traffic = JointDistribution(
    x_values=["sunny", "rainy"],
    y_values=["light", "heavy"],
    probabilities=[
        [0.40, 0.10],
        [0.20, 0.30],
    ],
    x_name="Weather",
    y_name="Traffic",
)

In [9]:
weather_given_heavy = weather_traffic.conditional(
    target="x",
    given_value="heavy",
)

print(weather_given_heavy.values)
print(weather_given_heavy.probabilities)

['sunny' 'rainy']
[0.25 0.75]
